\begin{align*}
p(x|y=j) = \frac{1}{(2\pi)^{n/2} |\Sigma_j|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x-\mu_j)\right)\\
p(y=j)=\phi_j\\
\phi_j=\frac{1}{m}\sum_{i=1}^{m}1\{y^{(i)}=j\}\\
\mu_j=\frac{\sum_{i=1}^{m}1\{y^{(i)}=j\}x^{(i)}}{\sum_{i=1}^{m}1\{y^{(i)}=j\}}\\
\Sigma_j = \frac{\sum_{i=1}^{m}1\{y^{(i)}=j\}(x^{(i)}-\mu_j)(x^{(i)}-\mu_j)^T}{\sum_{i=1}^{m}1\{y^{(i)}=j\}}
\end{align*}
At prediction we choose class with largest $p(y=j|x)=p(x|y=j)p(y=j)$

In [1]:
import numpy as np
import pandas as pd

In [2]:
class GDA:
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        m,n=self.X.shape
        self.classes=np.unique(y)
        self.n_classes=len(self.classes)
        self.mus=np.zeros((self.n_classes,n))
        self.covs = np.zeros((self.n_classes, n, n))
        self.phis=np.zeros(self.n_classes)
        for i,c in enumerate(self.classes):
            self.phis[i]=len(y[y==c])/m
            self.mus[i,:]=np.mean(self.X[y==c],axis=0)
            self.covs[i, :, :] = np.cov(self.X[y==c].T, bias=True)
    def predict(self,X):
        X=np.asarray(X)
        probs=np.zeros((X.shape[0],self.n_classes))
        for i,c in enumerate(self.classes):
            cov = self.covs[i]
            term_1 = -0.5 * X.shape[1] * np.log(2 * np.pi) - 0.5 * np.log(np.linalg.det(cov))
            diff = X - self.mus[i]
            term_2 = -0.5 * np.sum(diff@np.linalg.inv(cov) * diff, axis=1)
            probs[:,i]=term_1+term_2+np.log(self.phis[i])
        return self.classes[np.argmax(probs,axis=1)]

In [3]:
df=pd.read_csv('data/iris.csv')

In [4]:
X,y=df.loc[:,['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']],df.loc[:,'Species']

In [5]:
y=y.map({'Iris-setosa':0,'Iris-versicolor':1,'Iris-virginica':2})

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y)

In [7]:
X_train_std=X_train.std(axis=0)
X_train_mean=X_train.mean(axis=0)
X_train=(X_train-X_train_mean)/X_train_std
X_test=(X_test-X_train_mean)/X_train_std

In [8]:
gda=GDA()
gda.fit(X_train.values,y_train.values)
preds=gda.predict(X_test.values)
print(f"Accuracy: {np.mean(preds==y_test)*100}%")

Accuracy: 100.0%
